# Exploratory Data Analysis — LOATO-Bench

**Sprint 1A: Data Quality Assessment & Taxonomy Validation**

This notebook performs comprehensive EDA on the unified dataset (~20K+ samples from 5 sources) to inform Sprint 2A decisions:

1. **Dataset Overview**: Sample counts, sources, class balance
2. **Text Properties**: Length distributions, outliers, vocabulary
3. **GenTel Quality Gate**: Injection confidence scoring, filtering recommendations
4. **Taxonomy Validation**: Category coverage, merge recommendations
5. **Split Feasibility**: LOATO, direct→indirect, cross-lingual experiments
6. **Key Findings & Recommendations**

## 1. Setup & Imports

In [ ]:
# Standard library
import json

# Data & viz
import pandas as pd
import seaborn as sns

# LOATO-Bench modules
from loato_bench.analysis import eda, quality, visualization
from loato_bench.data import taxonomy
from loato_bench.utils.config import RESULTS_DIR
from loato_bench.utils.reproducibility import seed_everything

# Configure
seed_everything(42)
sns.set_style("whitegrid")
pd.set_option("display.max_columns", None)

print("✓ Imports complete")

## 2. Load Data

In [ ]:
# Load unified harmonized dataset
df = eda.load_unified_dataset()

print(f"Dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head()

## 3. Dataset Overview

In [ ]:
# Compute aggregate statistics
stats = eda.compute_dataset_statistics(df)

print("=== Dataset Statistics ===")
print(f"Total samples: {stats['total_samples']:,}")
print(f"Number of sources: {stats['num_sources']}")
print(f"Number of languages: {stats['num_languages']}")
print("\nClass balance:")
for label, count in stats["class_balance"].items():
    pct = count / stats["total_samples"] * 100
    label_name = "Benign" if label == 0 else "Injection"
    print(f"  {label_name} ({label}): {count:,} ({pct:.1f}%)")

print(f"\nIndirect injections: {stats['indirect_count']:,}")
print(f"Attack category coverage: {stats['attack_category_coverage']:.1f}%")

In [ ]:
# Source distribution
print("\n=== Source Distribution ===")
for source, count in stats["sources"].items():
    print(f"{source:20s}: {count:6,} samples")

## 4. Text Properties

In [ ]:
# Analyze text properties
text_props = eda.analyze_text_properties(df)

print("=== Text Length Statistics (characters) ===")
print(f"Min:    {text_props['char_lengths']['min']:,}")
print(f"Max:    {text_props['char_lengths']['max']:,}")
print(f"Mean:   {text_props['char_lengths']['mean']:.1f}")
print(f"Median: {text_props['char_lengths']['median']:.1f}")
print(f"Std:    {text_props['char_lengths']['std']:.1f}")

print("\n=== Outliers ===")
print(f"Very short (<10 chars): {text_props['outliers_short']:,}")
print(f"Very long (>5000 chars): {text_props['outliers_long']:,}")
print(f"Empty text: {text_props['empty_count']:,}")

## 5. GenTel Quality Gate

GenTel-Bench has 177K samples labeled "injection" but categories are **content harm types** (hate speech, violence) NOT injection techniques. We need to filter to genuine injection samples.

In [ ]:
# Detect GenTel quality issues
gentel_issues = quality.detect_gentel_quality_issues(df)

print("=== GenTel Quality Analysis ===")
print(f"GenTel sample count: {gentel_issues['gentel_count']:,}")
print("\nConfidence distribution:")
print(f"  Low (<0.3):    {gentel_issues['low_confidence_count']:,}")
print(f"  Medium (0.3-0.7): {gentel_issues['medium_confidence_count']:,}")
print(f"  High (≥0.7):   {gentel_issues['high_confidence_count']:,}")
print(f"\nMean score: {gentel_issues['mean_score']:.3f}")
print(f"Median score: {gentel_issues['median_score']:.3f}")

print("\n=== Issues Detected ===")
for issue in gentel_issues["issues_detected"]:
    print(f"  • {issue}")

In [ ]:
# Get filtering recommendation
gentel_filter = quality.recommend_gentel_filtering(df, threshold=0.4, max_samples=5000)

print("=== GenTel Filtering Recommendation ===")
print(gentel_filter["recommendation"])

## 6. Taxonomy Validation (Tier 1 + 2)

Apply Tier 1 (source mapping) and Tier 2 (regex patterns) to validate draft taxonomy categories.

In [ ]:
# Apply taxonomy mapping
df_mapped = taxonomy.apply_taxonomy_mapping(df)

# Compute coverage
coverage = taxonomy.compute_category_coverage(df_mapped)

print("=== Taxonomy Coverage ===")
print(f"Total injection samples: {coverage['total_injection_samples']:,}")
print(f"Mapped: {coverage['mapped_count']:,} ({coverage['coverage_percentage']:.1f}%)")
print(f"Unmapped: {coverage['unmapped_count']:,}")

print("\n=== Category Distribution ===")
for cat, count in sorted(coverage["category_counts"].items(), key=lambda x: x[1], reverse=True):
    pct = count / coverage["total_injection_samples"] * 100
    print(f"{cat:30s}: {count:5,} ({pct:5.1f}%)")

In [ ]:
# Check for small categories needing merges
merge_recs = taxonomy.recommend_category_merges(df_mapped, min_size=50)

print("=== Merge Recommendations ===")
for rec in merge_recs["merge_recommendations"]:
    print(rec)

print(f"\nCategories to keep ({len(merge_recs['categories_to_keep'])}):")
for cat in merge_recs["categories_to_keep"]:
    print(f"  • {cat}")

## 7. Split Feasibility Analysis

Validate that we have enough samples for each experiment type.

In [ ]:
# LOATO feasibility: need ≥200 samples per category
print("=== LOATO Split Feasibility ===")
loato_viable = []
loato_insufficient = []
for cat, count in coverage["category_counts"].items():
    if count >= 200:
        loato_viable.append((cat, count))
    else:
        loato_insufficient.append((cat, count))

print(f"Viable categories ({len(loato_viable)}):")
for cat, count in loato_viable:
    print(f"  ✓ {cat}: {count:,} samples")

if loato_insufficient:
    print(f"\nInsufficient categories ({len(loato_insufficient)}):")
    for cat, count in loato_insufficient:
        print(f"  ✗ {cat}: {count:,} samples (need 200)")

In [ ]:
# Direct → Indirect feasibility: need ≥500 indirect samples
indirect_count = df["is_indirect"].sum()
print("\n=== Direct → Indirect Feasibility ===")
print(f"Indirect samples: {indirect_count:,}")
if indirect_count >= 500:
    print("✓ Sufficient for direct→indirect experiment")
else:
    print(f"✗ Insufficient (need 500, have {indirect_count})")

In [ ]:
# Cross-lingual feasibility: need ≥300 per language
lang_dist = eda.analyze_language_distribution(df)
print("\n=== Cross-lingual Feasibility ===")
print(f"Non-English samples: {lang_dist['non_english_count']:,}")
print("\nLanguage distribution:")
for lang, count in sorted(lang_dist["counts"].items(), key=lambda x: x[1], reverse=True)[:10]:
    status = "✓" if count >= 300 else "✗"
    print(f"  {status} {lang}: {count:,}")

## 8. Visualizations

Create all EDA plots for thesis appendix.

In [ ]:
# Create output directory
figures_dir = RESULTS_DIR / "eda" / "figures"
figures_dir.mkdir(parents=True, exist_ok=True)

# Generate dashboard
saved_figures = visualization.create_eda_dashboard(df_mapped, figures_dir)
print(f"Created {len(saved_figures)} figures in {figures_dir}")

In [ ]:
# Display figures inline
from IPython.display import Image, display

for fig_path in saved_figures:
    print(f"\n{fig_path.name}")
    display(Image(filename=str(fig_path)))

## 9. Data Integrity Check

In [ ]:
# Validate data integrity
warnings = quality.validate_data_integrity(df)

print("=== Data Integrity Warnings ===")
for warning in warnings:
    print(f"  • {warning}")

## 10. Key Findings & Recommendations

### GenTel Filtering Decision
- **Current state**: [Insert findings]
- **Recommendation**: [Insert recommendation]

### Taxonomy Refinement
- **Coverage**: [Insert coverage %]
- **Merge plan**: [Insert categories to merge]

### Split Feasibility
- **LOATO**: [Insert viable categories]
- **Direct→Indirect**: [Insert status]
- **Cross-lingual**: [Insert status]

### Action Items for Sprint 2A
1. Apply GenTel filtering (threshold=0.4, cap=5K)
2. Merge small categories as recommended
3. Implement Tier 3 (LLM) taxonomy for unmapped samples
4. Generate stratified splits for all experiments

## 11. Export Results

In [ ]:
# Save reports to JSON
output_dir = RESULTS_DIR / "eda"
output_dir.mkdir(parents=True, exist_ok=True)

# GenTel quality report
with open(output_dir / "gentel_quality_report.json", "w") as f:
    json.dump({"issues": gentel_issues, "filtering": gentel_filter}, f, indent=2)

# Taxonomy coverage report
with open(output_dir / "taxonomy_coverage.json", "w") as f:
    json.dump({"coverage": coverage, "merges": merge_recs}, f, indent=2)

# Data integrity report
with open(output_dir / "data_integrity.json", "w") as f:
    json.dump({"warnings": warnings}, f, indent=2)

print(f"✓ Reports saved to {output_dir}")